# Quickstart — `oa_pipeline` end to end

This tutorial runs the full notebook based ocean acidification preprocessing
pipeline on a small deterministic synthetic dataset and shows what the
analyst facing output looks like.

**Time:** roughly 1 to 2 minutes once the package and Papermill are installed.

**What you will see:**

1. How to verify you are running from the project root.
2. How to regenerate the example workbook.
3. How to run the full pipeline with `run_pipeline.sh`.
4. How to open the final `analysis_ready.csv`.
5. How to inspect expected `PASS`, `REVIEW`, and `FAIL` outcomes.
6. Where to find reports, manifests, tables, and executed notebooks.


## 0. Prerequisites

Run this notebook from the project root, meaning the directory that contains:

```text
pyproject.toml
run_pipeline.sh
notebooks/
src/oa_pipeline/
examples/
```

Install the package and optional development dependencies first:

```bash
python -m pip install -e ".[all]"
```

On Windows, `run_pipeline.sh` requires Git Bash or WSL. The helper below
searches for `bash` on `PATH` and also checks the common Git Bash locations.


## 1. Imports, environment checks, and helper functions

This cell fails early if the notebook is opened from the wrong directory or
if the package is not importable in the active kernel.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
print(f"Project root: {PROJECT_ROOT}")

required_paths = [
    PROJECT_ROOT / "pyproject.toml",
    PROJECT_ROOT / "run_pipeline.sh",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "src" / "oa_pipeline",
    PROJECT_ROOT / "examples" / "make_example_data.py",
]

missing = [p for p in required_paths if not p.exists()]
assert not missing, (
    "This notebook should be run from the project root. Missing:\n"
    + "\n".join(str(p) for p in missing)
)

assert importlib.util.find_spec("oa_pipeline") is not None, (
    "oa_pipeline is not importable. Run:\n"
    "python -m pip install -e \".[all]\"\n"
    "from the project root, then restart this notebook kernel."
)

EXAMPLE_XLSX = PROJECT_ROOT / "examples" / "example_data.xlsx"
EXAMPLE_GEN = PROJECT_ROOT / "examples" / "make_example_data.py"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "quickstart"


def find_bash() -> str:
    """Find bash for running run_pipeline.sh."""
    candidate = shutil.which("bash")
    if candidate:
        return candidate

    windows_candidates = [
        Path("C:/Program Files/Git/bin/bash.exe"),
        Path("C:/Program Files/Git/usr/bin/bash.exe"),
        Path("C:/Program Files (x86)/Git/bin/bash.exe"),
        Path("C:/Program Files (x86)/Git/usr/bin/bash.exe"),
    ]

    for path in windows_candidates:
        if path.exists():
            return str(path)

    raise RuntimeError(
        "Could not find bash. Install Git Bash or run the pipeline "
        "from Git Bash / WSL."
    )


def id_column(df: pd.DataFrame) -> str:
    """Return the stable row identifier column in the final output."""
    if "record_id" in df.columns:
        return "record_id"
    if "sample_tag" in df.columns:
        return "sample_tag"
    raise KeyError("Neither record_id nor sample_tag found in final output.")


bash_exe = find_bash()
print(f"Using bash: {bash_exe}")
print(f"Python: {sys.executable}")


## 2. Generate or regenerate the example dataset

The example dataset is synthetic but realistic. It contains:

```text
20 sample rows
4 CRM rows
3 TRIS pH standard rows
```

The generator uses a fixed random seed, so the workbook is deterministic.

The final analysis table later in the workflow contains **sample rows only**.
CRM and pH standard rows are used for QC and excluded from the final sample
analysis table.


In [ ]:
# Regenerate the deterministic example workbook.
# This keeps the quickstart aligned with the current generator and tests.
result = subprocess.run(
    [sys.executable, str(EXAMPLE_GEN), "--out", str(EXAMPLE_XLSX)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

peek = pd.read_excel(EXAMPLE_XLSX, sheet_name="oa_data")
print(f"Loaded {len(peek)} rows and {len(peek.columns)} columns from {EXAMPLE_XLSX.name}")
display(peek.head())


## 3. Run the full pipeline

The runner script `run_pipeline.sh` executes the notebook chain and wires each
stage output into the next stage input.

This quickstart writes to:

```text
outputs/quickstart/
```

The cell below deletes only that fixed quickstart output folder before rerunning.


In [ ]:
# Guard against accidental deletion if OUTPUT_ROOT is edited.
assert OUTPUT_ROOT.resolve().is_relative_to((PROJECT_ROOT / "outputs").resolve()), (
    f"Refusing to delete outside the project outputs folder: {OUTPUT_ROOT}"
)

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

runner = PROJECT_ROOT / "run_pipeline.sh"
assert runner.exists(), f"Cannot find {runner}"

result = subprocess.run(
    [bash_exe, str(runner), str(EXAMPLE_XLSX), str(OUTPUT_ROOT)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print("Runner stdout tail:")
print("\n".join(result.stdout.splitlines()[-40:]))

if result.stderr.strip():
    print("\nRunner stderr tail:")
    print("\n".join(result.stderr.splitlines()[-40:]))

assert result.returncode == 0, (
    f"Pipeline failed with return code {result.returncode}. "
    "See stdout and stderr above."
)


## 4. Load the final analysis ready CSV

`analysis_ready.csv` is the analyst facing deliverable. It contains the
sample rows only; CRM and pH standard rows were used for correction and QC
and are not included in this final sample analysis table.

The pipeline is additive: it preserves rows and adds audit columns rather
than silently deleting questionable records.


In [ ]:
final_csv = OUTPUT_ROOT / "oa_stage4_outputs" / "data" / "analysis_ready.csv"
assert final_csv.exists(), f"Stage 4 did not produce {final_csv}"

ar = pd.read_csv(final_csv)
id_col = id_column(ar)

print(f"analysis_ready.csv rows: {len(ar)}")
print(f"analysis_ready.csv columns: {len(ar.columns)}")
print(f"Identifier column: {id_col}")

print("\nVerdict distribution:")
print(ar["analysis_audit_status"].value_counts().to_string())

display(ar.head())


### Expected verdicts for deliberately injected rows

The example workbook intentionally includes four sample row issues:

| Row | Injected issue | Expected status | Expected reason |
|---|---|---|---|
| S005 | salinity = 50, above `sal_max` | REVIEW | `range_flag` |
| S007 | missing `sample_id` | FAIL | `missing_key` |
| S010 | DIC species sum off by 200 µmol kg⁻¹ | FAIL | `strict_dic_species_fail` |
| S015 | negative HCO₃, physically impossible | FAIL | `strict_dic_species_fail` |

Most remaining rows should be PASS. Depending on the current replicate
conflict example and thresholds, there may be one or two additional REVIEW
rows. The key check is that the injected rows receive the expected status
and reason codes.


In [ ]:
manifest_path = OUTPUT_ROOT / "oa_stage4_outputs" / "logs" / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

reason_counts = pd.Series(
    manifest.get("reason_code_counts", {}),
    name="count",
).sort_values(ascending=False)

print("Reason codes that fired in Stage 4:")
if reason_counts.empty:
    print("(none)")
else:
    print(reason_counts.to_string())


## 5. Inspect the deliberately broken rows

This table extracts the expected non PASS rows from the final deliverable.
It uses `record_id` when available and falls back to `sample_tag` for older
output versions.


In [ ]:
broken_tags = ["S005", "S007", "S010", "S015"]

key_cols = [
    c for c in [
        id_col,
        "sample_id",
        "salinity",
        "co2aq_calc_umol_kg",
        "hco3_calc_umol_kg",
        "co3_calc_umol_kg",
        "dic_best_umol_kg",
        "analysis_audit_status",
        "analysis_audit_reason_codes",
    ]
    if c in ar.columns
]

display(ar.loc[ar[id_col].isin(broken_tags), key_cols])


## 6. Inspect one row's audit trail

Pick the salinity out of range row, `S005`. The pipeline keeps flag columns
from each stage, so you can trace what fired and where.


In [ ]:
row = ar.loc[ar[id_col] == "S005"].iloc[0]

flag_cols = [
    c for c in ar.columns
    if c.startswith("flag_") or c.startswith("flag_audit_")
]

fired = row[flag_cols][row[flag_cols] == True]

print("Flags that fired for S005:")
for col in fired.index:
    print(f"  {col}")

range_long_path = OUTPUT_ROOT / "oa_stage4_outputs" / "tables" / "range_flags_long.csv"
range_long = pd.read_csv(range_long_path)

print("\nS005 range violations from range_flags_long.csv:")
if id_col in range_long.columns:
    display(range_long[range_long[id_col].astype(str).eq("S005")])
else:
    # Fall back to sample_id text, which should still identify S005.
    display(
        range_long[
            range_long.get("sample_id", pd.Series(dtype="string"))
            .astype(str)
            .str.contains("OA-2024-005", na=False)
        ]
    )


## 7. Where to look next

Every major stage writes the same four kinds of output under
`outputs/quickstart/`:

```text
outputs/quickstart/
    oa_prelim_data__qc_outputs/   # Notebook 02
    oa_stage1a_outputs/           # Notebook 04
    oa_stage1b_outputs/           # Notebook 05
    oa_stage2_outputs/            # Notebook 06
    oa_stage3_outputs/            # Notebook 07
    oa_stage4_outputs/            # Notebook 08
        data/      analysis_ready.csv
        tables/    range_flags_long.csv, dic_species_audit.csv, ...
        reports/   report.md
        logs/      manifest.json, effective_config.json
```

The project level `runs/<UTC-timestamp>/` folder contains the fully executed
Papermill notebooks for the audit trail.

When you have a real dataset:

1. Replace `EXAMPLE_XLSX` with your workbook path.
2. Run the pipeline into a new output folder.
3. Inspect each stage's `reports/report.md`.
4. Check `oa_stage4_outputs/logs/manifest.json` for final row counts and reason codes.


In [ ]:
print("Per stage manifest excerpts:")

for stage_dir in sorted(OUTPUT_ROOT.iterdir()):
    manifest_file = stage_dir / "logs" / "manifest.json"
    if not manifest_file.exists():
        continue

    m = json.loads(manifest_file.read_text(encoding="utf-8"))
    rc = m.get("row_counts", {})

    print(f"\n  {stage_dir.name}/logs/manifest.json")
    print(f"    notebook:   {m.get('notebook', 'n/a')}")
    print(f"    rows:       {rc.get('n_rows', rc.get('staged_rows', 'n/a'))}")
    print(f"    parquet:    {m.get('parquet_written', 'n/a')}")


## 8. Optional next checks

Run these from a terminal at the project root:

```bash
python -m pytest -q
python -m pytest tests/test_pipeline_e2e.py -q
```

To run the same pipeline from a terminal:

```bash
./run_pipeline.sh examples/example_data.xlsx outputs/quickstart
```

On Windows PowerShell:

```powershell
bash .\run_pipeline.sh examples\example_data.xlsx outputs\quickstart
```
